# PrDiMP50 Tracker Training on GOT-10k

This notebook provides a complete training pipeline for the PrDiMP50 tracker with ResNet50 backbone.

**Requirements:**
- GOT-10k dataset
- PyTorch
- torchvision
- PIL
- matplotlib
- numpy

In [ ]:
!git clone https://github.com/heller007/PrDimp_unofficial.git

In [ ]:
import os

# Replace 'your-repo-name' with the actual name of the folder
repo_name = 'PrDimp_unofficial'

os.chdir(f'/kaggle/working/{repo_name}')

print("Current working directory is now:", os.getcwd())

In [ ]:
!git pull

## 1. Setup and Imports

In [ ]:
# Example (adjust torch/cu versions to your environment)
!pip install torch torchvision tqdm matplotlib pillow

# Optional: tensorboard
!pip install tensorboard

# If running in Colab and you need a specific torchvision:
# pip install --upgrade torchvision


In [ ]:
# Install required packages (uncomment if needed)
# !pip install torch torchvision pillow matplotlib numpy tqdm

import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from datetime import datetime

# Add project root to path
project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
# PrDiMP50 Integration & Training Notebook

This notebook connects the `data/` and `src/` modules in this workspace, performs quick sanity checks,
and runs a short training and evaluation cycle on GOT-10k.

Files expected (from workspace):
- data/got10k_dataset.py
- src/resnet50.py
- src/cls_feature_head.py
- src/initializer.py
- src/optimizer.py
- src/iou_head.py
- src/regression.py
- src/prdimp50.py
- src/prdimp_tracker.py
- src/online_update.py
- train_prdimp.py
- eval_got10k.py

Adjust `DATA_ROOT` to point to your GOT-10k dataset before training.


In [ ]:
# Cell 2 — setup environment and paths
import os, sys
PROJECT_ROOT = os.getcwd()  # change if notebook is opened from another folder
print("PROJECT_ROOT =", PROJECT_ROOT)

# Add project folders to PYTHONPATH so we can import src and data modules
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
# Ensure src and data directories are importable as packages
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
print("SRC_DIR =", SRC_DIR)
print("DATA_DIR =", DATA_DIR)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
if DATA_DIR not in sys.path:
    sys.path.insert(0, DATA_DIR)

# Device
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Path to GOT-10k dataset (CHANGE this to your path)
DATA_ROOT = "/kaggle/input/got10k"   # <- EDIT THIS cell before running training
print("DATA_ROOT (edit me) =", DATA_ROOT)

# Paths to the papers (local files we uploaded)
PAPER_PRDIMP = "/mnt/data/Danelljan_Probabilistic_Regression_for_Visual_Tracking_CVPR_2020_paper.pdf"
PAPER_DIMP   = "/mnt/data/Bhat_Learning_Discriminative_Model_Prediction_for_Tracking_ICCV_2019_paper.pdf"
print("PrDiMP paper path:", PAPER_PRDIMP)
print("DiMP paper path:", PAPER_DIMP)


In [ ]:
# Cell 3 - install required python packages (run once)
# Adjust for your environment (CUDA versions etc). If packages are already installed, pip will skip.
!pip install --upgrade pip
!pip install torch torchvision tqdm pillow matplotlib
# optional: tensorboard
!pip install tensorboard

In [ ]:
# Cell 4 — Import project modules (will raise error if import paths wrong)
# From the workspace screenshot your code is under src/ and data/ - import accordingly.

from importlib import reload
import data.got10k_dataset as got10k_dataset   # module: data/got10k_dataset.py
import src.resnet50 as resnet50_mod
import src.cls_feature_head as cls_head_mod
import src.initializer as initializer_mod
import src.optimizer as optimizer_mod
import src.iou_head as iou_head_mod
import src.regression as regression_mod
import prdimp50 as prdimp50_mod
import prdimp_tracker as prdimp_tracker_mod
import online_update as online_update_mod

print("Modules loaded successfully.")
print("GOT10k module:", got10k_dataset)
print("PrDiMP model module:", prdimp50_mod)


In [ ]:
# Cell 5 — small dataloader test and visualize template & search crops + GT
from torch.utils.data import DataLoader
from data.got10k_dataset import GOT10kTrainDataset, got10k_collate
from torchvision.transforms import functional as TF
import matplotlib.pyplot as plt
from PIL import ImageDraw, Image

# create dataset instance (use a small subset if you want)
ds = GOT10kTrainDataset(root_dir=DATA_ROOT, split="train",
                        template_size=(128,128), search_size=(320,320),
                        template_jitter=0.3, search_jitter=1.0, rng_seed=123)

dl = DataLoader(ds, batch_size=2, collate_fn=got10k_collate, num_workers=0)
batch = next(iter(dl))
print("Batch keys:", list(batch.keys()))
print("Shapes:", {k: (v.shape if hasattr(v, 'shape') else type(v)) for k,v in batch.items()})

# visualize first sample
tpl = TF.to_pil_image(batch['template'][0])
srch = TF.to_pil_image(batch['search'][0])
tpl_gt = batch['tpl_gt'][0].tolist()
srch_gt = batch['srch_gt'][0].tolist()

def draw_center_box(img, box, color='red', width=3):
    cx, cy, w, h = box
    x1 = cx - w/2; y1 = cy - h/2; x2 = cx + w/2; y2 = cy + h/2
    draw = ImageDraw.Draw(img)
    draw.rectangle([x1,y1,x2,y2], outline=color, width=width)
    return img

tpl_v = draw_center_box(tpl.copy(), tpl_gt, 'green')
srch_v = draw_center_box(srch.copy(), srch_gt, 'blue')
plt.figure(figsize=(10,5))
plt.subplot(1,2,1); plt.imshow(tpl_v); plt.title("Template"); plt.axis('off')
plt.subplot(1,2,2); plt.imshow(srch_v); plt.title("Search"); plt.axis('off')
plt.show()


In [ ]:
# Cell 6 — instantiate PrDiMP50 and run a small forward_train pass to check end-to-end shapes
from prdimp50 import PrDiMP50
model = PrDiMP50().to(device)
model.eval()
print("Model created on device:", device)

# use the batch from previous cell
template = batch['template'].to(device)
search   = batch['search'].to(device)
tpl_gt   = batch['tpl_gt'].to(device)
srch_gt  = batch['srch_gt'].to(device)

# Build template label maps in feature grid resolution for this small test
with torch.no_grad():
    tpl_feat = model.extract_classification_features(template)
    _, C, Hf, Wf = tpl_feat.shape
    tpl_label_maps = []
    for i in range(template.size(0)):
        cx, cy, _, _ = batch['tpl_gt'][i].tolist()
        # convert center from patch pixels to feature grid coords
        cx_f = cx / template.shape[3] * Wf
        cy_f = cy / template.shape[2] * Hf
        xs = torch.arange(Wf, device=device).view(1,1,1,Wf).float()
        ys = torch.arange(Hf, device=device).view(1,1,Hf,1).float()
        sigma = 2.0
        g = torch.exp(-((xs - cx_f)**2 + (ys - cy_f)**2)/(2*sigma*sigma))
        g = g / (g.sum() + 1e-12)
        tpl_label_maps.append(g)
    tpl_label_maps = torch.cat(tpl_label_maps, dim=0)  # (N,1,Hf,Wf)

# run forward_train (single batch)
out = model.forward_train(template_images=template,
                          search_images=search,
                          template_label_maps=tpl_label_maps,
                          search_gt_boxes=srch_gt.to(device))
print("Train forward outputs:", out)


In [ ]:
# Cell 7 (fixed) — robust helper to compute KL maps from score_map and label_pdf
import numpy as np
import matplotlib.pyplot as plt
import torch

def ensure_2d_grid(arr):
    """
    Accepts:
      - torch.Tensor or numpy array with shape one of:
         (H,W), (1,H,W), (1,1,H,W), (N,H,W), (N,1,H,W), (1,N,H,W) etc.
    Returns:
      - 2D numpy array (H,W) selecting the first sample / channel as needed.
    """
    # convert torch -> numpy
    if isinstance(arr, torch.Tensor):
        a = arr.detach().cpu().numpy()
    else:
        a = np.asarray(arr)

    # remove any singleton dims at front until we have 2D or until leftover dims = 2
    # but prefer to select first sample/channel if batch dimension exists.
    # Strategy: collapse leading dims >2 by selecting index 0 along them until a.ndim==2
    while a.ndim > 2:
        a = a[0]
    # Now a.ndim == 2
    return a

def compute_grid_pdf_from_score(score_t):
    """
    score_t: torch.Tensor or numpy array with shape like (1,1,H,W) or (1,H,W) or (H,W)
    returns (H,W) numpy predicted pdf (softmax over flattened grid).
    """
    s2d = ensure_2d_grid(score_t)   # (H,W)
    flat = s2d.reshape(-1).astype(np.float64)
    m = flat.max() if flat.size > 0 else 0.0
    ex = np.exp(flat - m)
    p = ex / (ex.sum() + 1e-12)
    return p.reshape(s2d.shape)

def compute_label_pdf_gaussian(H, W, center_xy, sigma=2.0):
    xs = np.arange(W)
    ys = np.arange(H)
    xs_g, ys_g = np.meshgrid(xs, ys)
    cx, cy = center_xy
    g = np.exp(-((xs_g - cx)**2 + (ys_g - cy)**2) / (2 * sigma * sigma))
    g = g / (g.sum() + 1e-12)
    return g

# Example using model's filter iterates (w0 vs final); make sure `model`, `template`, `search` exist
with torch.no_grad():
    tpl_feat = model.extract_classification_features(template.to(device))
    tpl_label = tpl_label_maps  # from earlier cell where you computed these
    filters = model.compute_filter_iterates(tpl_feat, tpl_label)
    w0 = filters[0]
    w_final = filters[-1]
    srch_feat = model.extract_classification_features(search.to(device))
    score_pre = model.apply_filter(srch_feat, w0)    # likely shape (1,1,Hf,Wf) or (N,1,Hf,Wf)
    score_post = model.apply_filter(srch_feat, w_final)

# convert to 2D pdfs robustly
pred_pre = compute_grid_pdf_from_score(score_pre)
pred_post = compute_grid_pdf_from_score(score_post)

# label gaussian centered in grid center
Hf, Wf = pred_pre.shape
label_pdf = compute_label_pdf_gaussian(Hf, Wf, center_xy=(Wf/2.0, Hf/2.0), sigma=2.0)

# KL maps: label * (log(label) - log(pred))
eps = 1e-12
kl_pre = label_pdf * (np.log(label_pdf + eps) - np.log(pred_pre + eps))
kl_post = label_pdf * (np.log(label_pdf + eps) - np.log(pred_post + eps))
kl_diff = kl_pre - kl_post   # positive means KL reduced at that cell (improvement)

print("KL before (sum):", kl_pre.sum(), "KL after (sum):", kl_post.sum(), "change:", kl_pre.sum()-kl_post.sum())

# visualize
plt.figure(figsize=(12,3))
plt.subplot(1,4,1); plt.imshow(pred_pre); plt.title("pred_pre"); plt.axis('off')
plt.subplot(1,4,2); plt.imshow(pred_post); plt.title("pred_post"); plt.axis('off')
plt.subplot(1,4,3); plt.imshow(label_pdf); plt.title("label_pdf"); plt.axis('off')
plt.subplot(1,4,4); plt.imshow(kl_diff); plt.title("KL improvement (pre - post)"); plt.axis('off')
plt.show()


In [ ]:
from train_prdimp import train_prdimp

train_prdimp(
    data_root=DATA_ROOT,
    save_dir="./checkpoints",
    batch_size=10,
    num_workers=4,
    num_epochs=50,
    lr=1e-4,
    device=device
)


In [ ]:
# Optionally run training from python (uncomment to use)
# from train.train_prdimp import train_prdimp
# train_prdimp(data_root=DATA_ROOT, save_dir="./checkpoints", batch_size=2, num_workers=2, lr=1e-4, num_epochs=1, device=device)


In [ ]:
# Cell 9 — run evaluation on validation split for the checkpoint you saved
# Replace the checkpoint path if different
CHECKPOINT="./checkpoints/prdimp50_epoch001.pth"
python eval_got10k.py --data_root "$DATA_ROOT" --model "$CHECKPOINT" --split val --device "$DEVICE"


In [ ]:
# from eval.eval_got10k import evaluate_got10k
# evaluate_got10k(model_path="./checkpoints/prdimp50_epoch001.pth", data_root=DATA_ROOT, split="val", device=device)


In [ ]:
# Cell 10 — run tracker on a short GOT-10k sequence and enable online updates
from data.got10k_dataset import GOT10kSequence
from src.prdimp_tracker import PrDiMPTracker
from src.online_update import OnlineUpdater

# Choose a sequence name from GOT-10k val (change to a real sequence name present in DATA_ROOT)
# To list sequences:
import os
seqs = sorted(os.listdir(os.path.join(DATA_ROOT, "val")))
print("Example sequences:", seqs[:5])
SEQ_NAME = seqs[0]  # change if desired

seq = GOT10kSequence(DATA_ROOT, "val", SEQ_NAME)
print("Sequence:", SEQ_NAME, "Frames:", len(seq))

model = PrDiMP50().to(device)
tracker = PrDiMPTracker(model, device=device)
# attach online updater
tracker.online_updater = OnlineUpdater(model, device=device)

# initialize
first_img, first_gt_center = seq[0]
x1 = first_gt_center[0] - first_gt_center[2]/2
y1 = first_gt_center[1] - first_gt_center[3]/2
init_box = [x1, y1, first_gt_center[2], first_gt_center[3]]
tracker.initialize(first_img, init_box)

# track first N frames
N = min(len(seq), 30)
preds = [init_box]
for i in range(1, N):
    img, gt = seq[i]
    pred = tracker.track(img)
    preds.append(pred)
    # optionally plot or print every few frames
    if i % 5 == 0:
        print(f"Frame {i}: pred={pred}")


In [ ]:
# Cell 11 — run tensorboard in notebook (if you logged events during training)
# %load_ext tensorboard
# %tensorboard --logdir runs
echo "Start tensorboard with: tensorboard --logdir runs"


In [ ]:
# Troubleshooting & tips

- If imports fail: ensure `PROJECT_ROOT`, `SRC_DIR`, and `DATA_DIR` are correctly inserted into `sys.path` (Cell 2).
- If dataset not found: set `DATA_ROOT` to the GOT-10k root containing `train/ val/ test/`.
- If GPU runs out of memory: use smaller batch size (Cell 8), reduce `proj_channels` or disable `proj` in backbone.
- If tracker drifts: tune `OnlineUpdater` thresholds in `src/online_update.py` (add_sample_thresh, update_interval).
- For faster iterations use `num_workers>0` and `torch.backends.cudnn.benchmark = True`.
